In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [5]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="رنگ مورد علاقه من سبز است.")]},
    {"configurable": {"thread_id": "1"}}
)

In [11]:
print(response["messages"][-1].content)

خیلی خوب! رنگ سبز با موفقیت ثبت شد.

اگر دوست دارید، می‌توانید کد رنگ HEX یا نام دقیق‌تری از سبز را هم مشخص کنید (مثلاً سبز زمردی #50C878 یا سبز olive #808000). همچنین می‌توانم این ترجیح را برای سایر تنظیمات یا پیشنهادها استفاده کنم.


In [13]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="سلام. حالت چه طوره؟")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

print(response["messages"][-1].content)

سلام! من خوبم، ممنون. تو چطور هستی؟ هر کمکی از دستم برمیاد بگو؛ می‌تونم کمک کنم به سوالات، ترجمه، نوشتن متن، پیدا کردن منابع، برنامه‌ریزی و یادگیری. دنبال چه چیزی هستی؟


## Read state

In [15]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [17]:
response = agent.invoke(
    { "messages": [HumanMessage(content="رنگ مورد علاقه من سبز است")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

خیلی خوب! رنگ مورد علاقه شما سبز است و با موفقیت ثبت شد.

آیا دوست دارید با لحاظ این ترجیح، پالت‌های رنگی سبز یا ترکیب‌های مناسب با سبز را پیشنهاد بدهم؟ همچنین اگر بخواهید می‌توانم رنگ فعلی را دوباره نشان بدهم.


In [19]:
response = agent.invoke(
    { "messages": [HumanMessage(content="رنگ مورد علاقه من چیه؟")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

رنگ مورد علاقه شما سبز است. آیا می‌خواهید پالت‌های رنگی سبز یا ترکیب‌های مناسب با سبز را پیشنهاد بدهم؟ یا بخواهید رنگ فعلی را دوباره نمایش بدهم؟


In [21]:
response

{'messages': [HumanMessage(content='رنگ مورد علاقه من سبز است', additional_kwargs={}, response_metadata={}, id='fe0da5a8-4307-4872-81c7-e2f023191bd6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 284, 'prompt_tokens': 165, 'total_tokens': 449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dqa7rJA8OaRoNeEwwQToICQJBO3uB', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec529-8c1e-73a0-9b29-8fa0be01f1c4-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'سبز'}, 'id': 'call_XkxcDynSdg2tYmUjDBJcJ6os', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_token